# Preparing Data for EDA – Extended Notebook

This notebook extends the recipes from **Chapter 2 – Preparing Data for EDA** of *Exploratory Data Analysis with Python Cookbook* (Packt).

It covers all core techniques plus practical enhancements:

1. Grouping data  
2. Appending data  
3. Concatenating data (horizontal)  
4. Merging data  
5. Sorting data  
6. Categorising / binning data  
7. Removing duplicates  
8. Dropping rows & columns  
9. Changing data formats  
10. Replacing values  
11. Dealing with missing values  
12. End-to-end preparation pipeline  

**Data**: synthetic Marketing Campaign dataset that mirrors the structure and first rows of the original Kaggle / book samples.


## 0. Imports and path setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the project scripts importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from data_preparation import (
    group_and_aggregate, group_mean,
    append_dataframes, concatenate_dataframes, merge_dataframes,
    sort_dataframe, categorize_numeric, categorize_by_quantiles,
    drop_duplicates, duplicate_report,
    drop_rows, drop_columns,
    change_dtype, to_numeric_safe,
    replace_values, map_values,
    missing_report, drop_missing, fill_missing,
    prepare_pipeline, print_audit,
)

DATA = ROOT / "data"
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("Ready. Data folder:", DATA)

## 1. Load and inspect the main dataset

In [ ]:
marketing_data = pd.read_csv(DATA / "marketing_campaign.csv")
# Keep the columns used in most book recipes
cols = ["ID", "Year_Birth", "Education", "Marital_Status", "Income",
        "Kidhome", "Teenhome", "Dt_Customer", "Recency",
        "NumStorePurchases", "NumWebVisitsMonth"]
marketing_data = marketing_data[[c for c in cols if c in marketing_data.columns]]
print("Shape:", marketing_data.shape)
marketing_data.head(2).T

In [ ]:
marketing_data.dtypes

## 2. Grouping data

**Book recipe**: average number of store purchases by number of kids at home.

**Extension**: multi-column grouping and multiple aggregations in one call.


In [ ]:
# Classic single aggregation (book style)
group_mean(marketing_data, "Kidhome", "NumStorePurchases")

In [ ]:
# Extended: multiple metrics
group_and_aggregate(
    marketing_data,
    by=["Education", "Kidhome"],
    agg={"NumStorePurchases": ["mean", "count"], "Income": "median"}
).head(10)

## 3. Appending data (vertical stack)

Combine two sample files that share the same schema.


In [ ]:
s1 = pd.read_csv(DATA / "marketing_campaign_append1.csv")
s2 = pd.read_csv(DATA / "marketing_campaign_append2.csv")
# Align to common columns
common = list(set(s1.columns) & set(s2.columns) & set(marketing_data.columns))
s1, s2 = s1[common], s2[common]
print("Sample 1:", s1.shape, "Sample 2:", s2.shape)

appended = append_dataframes([s1, s2], ignore_index=True)
print("Appended:", appended.shape)
appended.head(3)

## 4. Concatenating data (horizontal)

Join feature sets that share the same row order / index.


In [ ]:
c1 = pd.read_csv(DATA / "marketing_campaign_concat1.csv")
c2 = pd.read_csv(DATA / "marketing_campaign_concat2.csv")
print("Left:", c1.shape, "Right:", c2.shape)

concatenated = concatenate_dataframes([c1, c2], axis=1)
print("Concatenated:", concatenated.shape)
concatenated.head(3)

## 5. Merging data

SQL-style join on a common key (`ID`).


In [ ]:
m1 = pd.read_csv(DATA / "marketing_campaign_merge1.csv")
m2 = pd.read_csv(DATA / "marketing_campaign_merge2.csv")
merged = merge_dataframes(m1, m2, on="ID", how="inner", indicator=True)
print("Merged:", merged.shape)
print(merged["_merge"].value_counts())
merged.head(3)

## 6. Sorting data


In [ ]:
sorted_data = sort_dataframe(marketing_data, by="NumStorePurchases", ascending=False)
sorted_data[["ID", "NumStorePurchases"]].head(8)

In [ ]:
# Multi-key sort: high store purchases first, then most recent
sort_dataframe(
    marketing_data,
    by=["NumStorePurchases", "Recency"],
    ascending=[False, True]
)[["ID", "NumStorePurchases", "Recency"]].head(5)

## 7. Categorising / binning data

Two approaches shown in the book: explicit edges and automatic equal-width bins.


In [ ]:
# Explicit bins (book style)
marketing_data = marketing_data.copy()
marketing_data["purchase_level"] = categorize_numeric(
    marketing_data["NumStorePurchases"],
    bins=[0, 4, 8, 13],
    labels=["Low", "Moderate", "High"]
)
marketing_data[["NumStorePurchases", "purchase_level"]].head()

In [ ]:
# Equal-width bins
marketing_data["purchase_level_eq"] = categorize_numeric(
    marketing_data["NumStorePurchases"], bins=3, labels=["Low", "Moderate", "High"]
)
# Quantile-based (equal frequency) – extension
marketing_data["purchase_q"] = categorize_by_quantiles(
    marketing_data["NumStorePurchases"], q=3, labels=["Low", "Moderate", "High"]
)
marketing_data[["NumStorePurchases", "purchase_level", "purchase_level_eq", "purchase_q"]].head(8)

## 8. Removing duplicates


In [ ]:
subset = ["Education", "Marital_Status", "Kidhome", "Teenhome"]
print("Before:", marketing_data.shape)
print(duplicate_report(marketing_data, subset=subset))

deduped = drop_duplicates(marketing_data[subset], subset=subset)
print("After drop_duplicates on subset:", deduped.shape)
deduped.head()

## 9. Dropping rows and columns


In [ ]:
sample = marketing_data.head(5).copy()
print("Original sample:")
display(sample)

print("\nDrop row at index 1:")
display(drop_rows(sample, labels=[1]))

print("\nDrop column Year_Birth:")
display(drop_columns(sample, "Year_Birth"))

## 10. Changing data format / dtypes


In [ ]:
# Fill then cast Income to integer
tmp = fill_missing(marketing_data, strategy="constant", columns=["Income"], fill_value=0)
tmp = change_dtype(tmp, "Income", "int64")
print(tmp[["Income"]].dtypes)
tmp[["Income"]].head()

## 11. Replacing values


In [ ]:
replaced = replace_values(
    marketing_data,
    column="Teenhome",
    to_replace=[0, 1, 2],
    value=["has no teen", "has teen", "has teen"]
)
replaced[["Teenhome"]].value_counts()

## 12. Dealing with missing values


In [ ]:
print(missing_report(marketing_data).query("missing_count > 0"))

# Strategy 1: drop rows with any NA
clean_drop = drop_missing(marketing_data, how="any")
print("After dropna(how='any'):", clean_drop.shape)

# Strategy 2: fill numeric with median
clean_fill = fill_missing(marketing_data, strategy="median", columns=["Income"])
print("Remaining NAs after median fill:", clean_fill["Income"].isna().sum())

## 13. End-to-end preparation pipeline

A convenience function that chains several common steps and returns an audit log.


In [ ]:
cleaned, audit = prepare_pipeline(
    marketing_data,
    fill_strategy="median",
    fill_columns=["Income"],
    drop_duplicates_subset=["ID"],  # keep unique customers
    sort_by="Recency",
)
print_audit(audit)
cleaned.head(3)

## Next steps

- Copy `templates/data_prep_template.py` and adapt the CONFIG section for your own CSV.
- Import any function from `scripts/data_preparation.py` into production pipelines.
- See the summary report (`reports/Preparing_Data_for_EDA_Report.docx`) for a complete file map and usage guide.
